# Hoja de trabajo HT06

In [14]:
import numpy as np
from scipy import stats
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.outliers_influence import variance_inflation_factor
from patsy import dmatrices

### Parte 1: Preparación y exploración

In [15]:
#Verificación de correlaciones
vars_modelo = ['tip', 'total_bill', 'size']
matriz_corr_mult = df[vars_modelo].corr(method='pearson')

print("Matriz de Correlación:")
display(matriz_corr_mult)

Matriz de Correlación:


,tip,total_bill,size
tip,1.000000,0.675734,0.489299
total_bill,0.675734,1.000000,0.598315
size,0.489299,0.598315,1.000000


**Pregunta: ¿Qué correlación existe entre total_bill y size? ¿Esto podría representar un problema de multicolinealidad en el modelo múltiple? Justifique.**

Entre total_bill y size existe una correlación alrededor de 0.59, una correlación moderada. Es decir, a mayor cantidad de comensales en una mesa, mayor suele ser el consumo total de la cuenta, lo cual tiene lógica. 

Esto sí representa un problema de multicolinealidad, ya que dos variables independiente están fuertemente correlacionadas entre sí, lo que sugiere que comparten información superpuesta. Por ello, se debe analizar detalladamente. 

### Ajuste del Modelo de Regresión Lineal Múltiple

In [16]:
# tip ~ total_bill+size
modelo_ols = ols('tip ~ total_bill + size', data=df).fit()

# Extraer los coeficientes 
print(modelo_ols.summary())

#Extraer métricas
r_squared_mult = modelo_ols.rsquared
adj_r_squared_mult = modelo_ols.rsquared_adj
print(f"\nR² Múltiple: {r_squared_mult:.4f}")
print(f"R² Ajustado Múltiple: {adj_r_squared_mult:.4f}")

                            OLS Regression Results                            
Dep. Variable:                    tip   R-squared:                       0.468
Model:                            OLS   Adj. R-squared:                  0.463
Method:                 Least Squares   F-statistic:                     105.9
Date:                Tue, 27 Oct 2026   Prob (F-statistic):           9.67e-34
Time:                        12:08:22   Log-Likelihood:                -347.99
No. Observations:                 244   AIC:                             702.0
Df Residuals:                     241   BIC:                             712.5
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.6689      0.194      3.455      0.0

**Pregunta: ¿El R² aumentó al agregar size? ¿El R² ajustado también mejoró?**

Sí, el R² incrementa ligeramente en comparación con el modelo simple, lo que nos indica que añadir el tamaño de la mesa explica una pequeña fracción de la variabilidad de las propinas.

El R² ajustado casi no cambia. Como esta medida está, de alguna manera, castigada por incluir variables que no aportan suficiente información nueva, nos demuestra que meter el número de comensales casi no mejora la predicción del modelo.

### Parte 3: Detección de Multicolinealidad (VIF)

In [17]:
#Preparar las matrices
y_vif, X_vif = dmatrices('tip ~ total_bill + size', data=df, return_type='dataframe')

#Calculo del VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print(vif_data)

     Variable       VIF
0   Intercept  1.000000
1  total_bill  1.557586
2        size  1.557586


**Pregunta: Según los valores de VIF obtenidos, ¿existe evidencia de multicolinealidad grave entre total_bill y size? ¿Qué implicaría esto para la interpretación de los coeficientes?**

Al calcular el VIF, ambas variables arrojaron un valor cercano a 1.56, es decir, no existe evidencia de multicolinealidad grave entre ellas. Por lo tanto, los coeficientes de ambas variables se mantienen estables y pueden interprestarse con confianza y sin temor que la superposición distorsione los resultados y, por ende, la interpretación.

### Parte 4: Interpretación de Resultados

El modelo de regresión lineal múltiple estimado indica que tanto la cuenta total como el tamaño de la mesa influten en la propina, alcanando un coeficiente de determinación de 0.47, lo que significa que el 47% de la variabilidad de las propinas es explicada conjuntamente por estos factores. 

De tal forma que, al mantener constante el tamaño de la mesa, un aumento de $1 en la cuenta total se traduce en un aumento aproximado de 9.2 centavos en la propina. Por otro lado, controlando el efecto de la cuenta, cada comensal adicional incrementa la propina en unos 19 centavos, reflejando que las mesas más numerosas tienden a dejar mejor bonificación. 

En conclusión, los valores de significancia estadística confirman que ambas variables aportan información relevante al modelo del restaurante. De modo que, la gerencia puede prever mejores ingresos en mesas con mayor consumo y mayor volumen de personas. 

### Parte 5: Reflexión Crítica

**¿Por qué es importante revisar el VIF antes de interpretar los coeficientes de un modelo múltiple?**

Es importante revisar el VIF antes de interpretar los coeficiente de un modelo múltiple porque nos permite identificar si las variables independientes (predictoras) están midiendo el mismo fenómeno. Cuando dos predictores están altamente correlacionados, el modelo pierde estabilidad matemática, lo cual se traduce en que los errores estándar de los coeficientes se inflen artificialmente y que las estimaciones varíen drásticamente ante pequeños cambios en los datos.  

Esto puede provocar que una variable importante parezca estadísticamente no significativa cuando en realidad sí tiene un efecto real en la variable dependiente, cambiando por completo la interpretación.